# Analyze

This notebook is the page [Analyze](https://quarcs-lab.github.io/geo4bolivia/analyze.html) of the [geo4bolivia documentation](https://quarcs-lab.github.io/geo4bolivia/). Run the cells from the top.

This page takes one question through every family of analysis functions: **did sustainable development become more equal across Bolivia's municipalities between about 2012 and about 2024, and what role does space play?** Each step is one or two calls, read in a few sentences, with a link to the article that explains the method in full. The page is a tour, not a study: the [case study](https://quarcs-lab.github.io/geo4bolivia/articles/case-study-sdg.html) does the same work at full length.

Every function returns a **result object**. Displayed, it shows its main figure and a reading of the numbers; `.fig`, `.df`, `.gt` and `.interpret()` give the parts one by one ([Get started](https://quarcs-lab.github.io/geo4bolivia/get-started.html) explains them).

In [ ]:
!pip install -q "geo4bolivia[mgwr]"

The `[mgwr]` extra is needed only for the last step.

## The data

The dataset `sdg` holds the **Sustainable Development Index** (`index_sdg`, called IDS, from 0 to 100) of the SDG Atlas 2026 for the 343 municipalities, in two periods stored as the years 2012 and 2024, with the census `population` and the `urbanization` rate.

In [ ]:
import geo4bolivia as geo

ids = geo.load("sdg", groups=["department"])
cols = ["variable", "year", "count", "mean", "std", "min", "max"]
geo.describe(ids, variables=["index_sdg", "urbanization"])[cols]

The index rose by about ten points on average, from 50.1 to 60.1. `urbanization` exists only for 2024, so the models further down describe that year alone.

## Two periods, not two years
"2012" and "2024" are nominal: each of the 81 indicators behind the index was measured in its own years, near those two (`geo.reference_years("sdg03")` lists them for one goal). Rates "per year" below divide by the twelve years between the nominal dates and are approximate.

## 1. Spatial dependence

**Question: do municipalities with similar values lie next to each other?** Global Moran's I measures it for the whole country: it is close to zero when values are placed at random, and positive when neighbours are alike. `year="all"` computes it in every year.

In [ ]:
geo.moran(ids, "index_sdg", year="all").df

Moran's I is 0.43 in 2012 and 0.47 in 2024. In both periods none of the 999 random reshufflings of the values over the map gave a value as high (`p_sim` = 0.001, the smallest the test can return), so the index is clearly clustered, and no less in 2024 than in 2012.

Local Moran statistics (LISA) show *where*. Each municipality is compared with its neighbours and, when the pattern is unlikely under random placement, assigned to a type: High-High (a high value among high neighbours), Low-Low, or one of the two kinds of outlier.

In [ ]:
lisa = geo.lisa(ids, "index_sdg", year=2024)
lisa

Of the 343 municipalities, 60 are High-High and 45 Low-Low; 11 are outliers and 227 are not significant. The next table counts them by department.

In [ ]:
where = lisa.df.merge(ids.loc[ids["year"] == 2024, ["geo_id", "department"]])
where.groupby(["department", "cluster"]).size().unstack(fill_value=0)

The High-High municipalities are mostly in Santa Cruz (17), Cochabamba (14), La Paz (12) and Tarija (10, of its 11 municipalities); the Low-Low ones mostly in Potosí (13), La Paz (13) and Oruro (8). The map locates clusters; it does not say why they are there, and "high" and "low" are relative to the national mean of that year.

→ [Maps and spatial dependence](https://quarcs-lab.github.io/geo4bolivia/articles/spatial-dependence.html): class schemes, spatial weights, the permutation test, and how much the result depends on the definition of "neighbour".

## 2. Convergence

**Question: did the municipalities that started lower rise faster?** β-convergence regresses each municipality's average yearly growth of the index on the logarithm of its initial level. A negative slope means that lower starting points go with faster growth.

In [ ]:
beta = geo.beta_convergence(ids, "index_sdg")
beta

The slope is −0.029 (standard error 0.002), and the initial level accounts for 44% of the differences in growth (R² = 0.44). At that pace half of a gap between two municipalities would close in about 20 years. Two cautions: an index with a ceiling at 100 leaves more room to rise at the bottom, and a negative slope does not by itself mean that the spread of the index fell (step 4 checks that directly).

The other two convergence functions need more years than the Atlas has:

- `geo.sigma_convergence` estimates the *trend* of the dispersion over time, so it needs at least three years.
- `geo.convergence_clubs` follows each unit's path over time to sort units into groups that converge among themselves, so it needs at least six.

Two datasets have long yearly series: `pop` (gridded population, 2001–2020) and `ntl` (night-time lights per capita, 2012–2020), both for the 339 municipalities of the 2015 map (`map339`). Called on a dataset that is too short, either function says so and names these two.

→ [Convergence: β, σ and clubs](https://quarcs-lab.github.io/geo4bolivia/articles/convergence.html): all three methods, the spatial versions of β-convergence, and the long panels.

## 3. Transitions and mobility

**Question: did the clusters of step 1 persist, and how much did municipalities change position?** `geo.lisa_transitions` runs the LISA analysis in both periods, on the same units and neighbours, and counts the moves between types.

In [ ]:
moves = geo.lisa_transitions(ids, "index_sdg")
moves

Rows are the type in 2012 and columns the type in 2024. 275 of the 343 municipalities (80%) have the same type in both periods. Of the 46 High-High municipalities of 2012, 39 are still High-High; of the 44 Low-Low, 30 are still Low-Low and 13 are no longer significant. No municipality moved between High-High and Low-Low. Moves to and from "Not significant" can come from p-values near the threshold, so read them with care.

`geo.rank_mobility` ranks the municipalities in each period (rank 1 is the highest index) and compares the two rankings.

In [ ]:
ranks = geo.rank_mobility(ids, "index_sdg")
ranks

Each point is a municipality; on the diagonal it kept its rank. The Spearman correlation of the two rankings is 0.84 (1 would be an unchanged order), a municipality moved 40 positions on average (median 29), and 56% of them stayed in the same fifth of the ranking. The top fifth is the most stable (77% stayed). A rank is relative: a municipality can lose positions while its index rises, when others rose more.

→ [Transitions and mobility](https://quarcs-lab.github.io/geo4bolivia/articles/transitions.html): Markov and spatial Markov transition matrices, which need the `[dynamics]` extra, and these two functions in detail.

## 4. Inequality

**Question: how unequal are the municipalities, and is the inequality between departments or inside them?** `geo.inequality` computes the Gini and Theil indices in each year (both are 0 when every unit has the same value) and draws the Lorenz curve.

In [ ]:
ineq = geo.inequality(ids, "index_sdg")
ineq.gt

Both measures fell: the Gini index from 0.065 to 0.049 and the Theil index from 0.0066 to 0.0038. With two periods the two values can be compared, but a trend cannot be tested. The values are small because the index is bounded and most municipalities lie within a few points of each other; they are not comparable with the Gini of income or population.

Theil's index can be split exactly into inequality **between** groups (differences among department averages) and **within** them.

In [ ]:
theil = geo.theil_decomposition(ids, "index_sdg", by="department")
theil

Differences between the nine departments account for 20% of the inequality in 2012 and 15% in 2024. Most of the inequality between municipalities is found inside departments, and the part between departments shrank faster than the part within.

→ [Regional inequality](https://quarcs-lab.github.io/geo4bolivia/articles/inequality.html): what each measure is sensitive to, the Lorenz curve, trends on long panels, the spatial Gini, and what an inequality index of a bounded indicator can and cannot say.

## 5. Models

**Question: what goes with a higher index, and is the relationship the same once neighbours are taken into account?** `geo.regression` fits ordinary least squares, with several specifications side by side. Here the index of 2024 is regressed on the urbanization rate, first alone and then with population and department fixed effects (a separate intercept for each department).

In [ ]:
reg = geo.regression(ids, "index_sdg",
                     [["urbanization"], ["urbanization", "population"]],
                     fixed_effects=[None, "department"])
reg.gt

A municipality whose urbanization rate is ten percentage points higher has an index about 0.9 points higher (coefficient 0.094). The coefficient barely moves (0.087) when municipalities are compared only with others of the same department and population is held constant. Urbanization alone accounts for 31% of the variation of the index.

Least squares assumes that the errors of neighbouring municipalities are unrelated. `geo.spatial_diagnostics` tests that on the residuals and points to the spatial model that fits the pattern it finds.

In [ ]:
diag = geo.spatial_diagnostics(ids, "index_sdg", "urbanization")
diag

The residuals are as clustered as the index itself (Moran's I = 0.47), so urbanization does not account for the spatial pattern of step 1. Both robust tests reject, and the one for a spatial error model is the larger (23.1 against 4.5), which is the model the function recommends.

In [ ]:
sem = geo.spatial_model(ids, "index_sdg", "urbanization", model="error")
sem

In the spatial error model the coefficient of urbanization is 0.081, a little lower than the least-squares one, and the spatial parameter λ is 0.67: what the model leaves unexplained is strongly correlated between neighbours. The association with urbanization holds; the neighbourhood pattern has another source, which this model does not identify.

→ [Regression, panel and spatial models](https://quarcs-lab.github.io/geo4bolivia/articles/models.html): fixed effects and clustered errors, panel estimators, and the spatial lag and Durbin models with their direct and spillover estimates.

## 6. Spatial heterogeneity

**Question: is the association with urbanization the same everywhere?** The models of step 5 estimate one coefficient for the country. Multiscale geographically weighted regression (MGWR) fits a regression at every municipality, giving more weight to the nearest ones, so the coefficient becomes a surface that can be mapped. It also chooses how many neighbours to use (the *bandwidth*): a small bandwidth means an association that changes over short distances. The search takes about half a minute.

In [ ]:
mg = geo.mgwr(ids, "index_sdg", "urbanization")
mg.figs["urbanization"]

In [ ]:
local = mg.df.merge(ids.loc[ids["year"] == 2024, ["geo_id", "department"]])
print(mg.bw, "| R2 =", round(mg.r2, 3))
local.groupby("department")["urbanization_coef"].mean().round(2).sort_values()

MGWR standardizes the variables, so the coefficients are in standard deviations. The local coefficient of urbanization is positive everywhere, from 0.21 to 1.01, and significant in 326 of the 343 municipalities. It is estimated from the 90 nearest municipalities at each place, about a quarter of the country: a regional rather than a national pattern. On average it is highest in Potosí, Cochabamba and Chuquisaca (0.73 to 0.79) and lowest in Pando, Beni and Santa Cruz (0.32 to 0.38).

→ [Spatial heterogeneity: MGWR](https://quarcs-lab.github.io/geo4bolivia/articles/heterogeneity.html): bandwidths, the local tests, several regressors, and how to read the coefficient maps.

## What the tour found

All of these are associations in the data; none shows a cause.

- **More equal, by the measures used here.** Between about 2012 and about 2024 the index rose in 342 of the 343 municipalities, municipalities that started lower rose faster (β = −0.029), and the Gini and Theil indices of the index fell.
- **Clustered, and no less so.** Moran's I is 0.43 in 2012 and 0.47 in 2024. Four of five municipalities have the same LISA type in both periods, and none moved between a High-High and a Low-Low cluster.
- **A reshuffled middle, a stable top.** The ranking changed (rank correlation 0.84), mostly in its middle; 77% of the top fifth stayed there.
- **Mostly within departments.** Differences between department averages are 15% of the inequality in 2024, down from 20%.
- **Urbanization goes with a higher index, everywhere but not equally.** The coefficient is positive in the national models and in every local one, and larger in Potosí, Cochabamba and Chuquisaca than in Santa Cruz, Beni and Pando. It leaves the spatial clustering of the index unaccounted for.

So the gaps between municipalities narrowed while the geography of high and low values stayed largely in place. What keeps it in place is a question these descriptive tools raise and cannot answer.

## Where next

- **[Case study: the Sustainable Development Index](https://quarcs-lab.github.io/geo4bolivia/articles/case-study-sdg.html)**: the same question, at full length.
- **[Articles](https://quarcs-lab.github.io/geo4bolivia/articles/index.html)**: one article per method.
- **[Explore](https://quarcs-lab.github.io/geo4bolivia/explore.html)**: the first look at a variable, before any of this.
- **[Function reference](https://quarcs-lab.github.io/geo4bolivia/reference/index.html)**: every function, with an example you can paste into Colab.